# 09 — Time-Based Splitting
## 1. Objective and split boundary
Create deterministic chronological analytical partitions only; do not preprocess, engineer features, evaluate, or train models.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from urban_ops.cleaning.outputs import sha256_file
from urban_ops.splitting.pipeline import run_time_based_split

CONFIG_PATH = PROJECT_ROOT / 'configs/data/splits.yaml'
result = run_time_based_split(config_path=CONFIG_PATH)
TABLES = result.tables
result.metadata.split_id

'20260804T170340Z_9d945cb2da0eecfc'

## 2. Source cleaning run

In [2]:
pd.Series({
    'cleaning_run_id': result.source.metadata.cleaning_run_id,
    'eligible_path': str(result.source.eligible_path),
    'eligible_sha256': result.source.eligible_sha256,
    'eligible_rows': len(result.source.frame),
})

cleaning_run_id                    20260804T161733Z_560ff9d5f0dd38f7
eligible_path      /Users/mohammadmubashir/VCode/urban-operations...
eligible_sha256    8f3b24ae342cb32af726d4d191c00e181037d4ae01672c...
eligible_rows                                                  35960
dtype: object

## 3. Source-data verification

In [3]:
TABLES['split_integrity_checks.csv'].query("area == 'source'")

,check_id,area,status,observed_value,expected_value,affected_rows,message
0,source.non_empty,source,PASS,35960,> 0,0,Eligible source is non-empty.
1,source.identifiers_non_null,source,PASS,0,0,0,Identifiers are non-null.
2,source.identifiers_unique,source,PASS,0,0,0,Identifiers are unique.
3,source.timestamps_non_null,source,PASS,0,0,0,Split timestamps are non-null.


## 4. Why time-based splitting is required

In [4]:
'Random assignment is prohibited because future complaints must not inform evaluation on earlier complaints.'

'Random assignment is prohibited because future complaints must not inform evaluation on earlier complaints.'

## 5. Split timestamp decision

In [5]:
pd.Series({'timestamp_column': result.metadata.timestamp_column, 'interval': result.metadata.interval_convention, 'timezone': str(result.source.frame.created_date.dt.tz)})

timestamp_column              created_date
interval            left_closed_right_open
timezone                               UTC
dtype: str

## 6. Monthly row-count profile

In [6]:
TABLES['monthly_target_distribution.csv'][['month', 'row_count', 'cumulative_row_count', 'cumulative_row_share']]

,month,row_count,cumulative_row_count,cumulative_row_share
0,2024-01,1428,1428,0.039711
1,2024-02,1306,2734,0.076029
2,2024-03,1460,4194,0.116630
3,2024-04,1942,6136,0.170634
4,2024-05,1760,7896,0.219577
5,2024-06,1525,9421,0.261986
6,2024-07,1498,10919,0.303643
7,2024-08,1656,12575,0.349694
8,2024-09,1477,14052,0.390768
9,2024-10,1532,15584,0.433370


## 7. Monthly target-rate profile

In [7]:
TABLES['monthly_target_distribution.csv'][['month', 'on_time_count', 'missed_count', 'missed_target_rate', 'has_both_classes']]

,month,on_time_count,missed_count,missed_target_rate,has_both_classes
0,2024-01,659,769,0.538515,True
1,2024-02,662,644,0.493109,True
2,2024-03,770,690,0.472603,True
3,2024-04,1110,832,0.428424,True
4,2024-05,995,765,0.434659,True
5,2024-06,844,681,0.446557,True
6,2024-07,885,613,0.409212,True
7,2024-08,953,703,0.424517,True
8,2024-09,939,538,0.364252,True
9,2024-10,985,547,0.357050,True


## 8. Temporal instability review

In [8]:
TABLES['temporal_drift_summary.csv'].query("comparison_type == 'monthly_summary'")

,comparison_type,source_period,destination_period,source_target_rate,destination_target_rate,absolute_difference,relative_difference,source_rows,destination_rows,metric_value,interpretation
3,monthly_summary,minimum_monthly_target_rate,2025-11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.307754,Observed monthly instability; not an automatic...
4,monthly_summary,maximum_monthly_target_rate,2025-01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.610039,Observed monthly instability; not an automatic...
5,monthly_summary,monthly_target_rate_range,max minus min,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.302285,Observed monthly instability; not an automatic...
6,monthly_summary,largest_month_to_month_absolute_change,2024-12,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.215554,Observed monthly instability; not an automatic...


## 9. Candidate split boundaries

In [9]:
candidate = TABLES['candidate_split_boundaries.csv']
candidate[['candidate_id', 'train_start', 'train_end_exclusive', 'validation_start', 'validation_end_exclusive', 'test_start', 'test_end_exclusive']]

,candidate_id,train_start,train_end_exclusive,validation_start,validation_end_exclusive,test_start,test_end_exclusive
0,A,2024-01-01T00:00:00+00:00,2025-07-01T00:00:00+00:00,2025-07-01T00:00:00+00:00,2025-10-01T00:00:00+00:00,2025-10-01T00:00:00+00:00,2026-01-01T00:00:00+00:00
1,B,2024-01-01T00:00:00+00:00,2025-04-01T00:00:00+00:00,2025-04-01T00:00:00+00:00,2025-09-01T00:00:00+00:00,2025-09-01T00:00:00+00:00,2026-01-01T00:00:00+00:00
2,C,2024-01-01T00:00:00+00:00,2025-01-01T00:00:00+00:00,2025-01-01T00:00:00+00:00,2025-07-01T00:00:00+00:00,2025-07-01T00:00:00+00:00,2026-01-01T00:00:00+00:00


## 10. Candidate comparison

In [10]:
candidate[['candidate_id', 'train_rows', 'validation_rows', 'test_rows', 'train_missed_rate', 'validation_missed_rate', 'test_missed_rate', 'maximum_rate_difference', 'qualified', 'rank', 'recommended']]

,candidate_id,train_rows,validation_rows,test_rows,train_missed_rate,validation_missed_rate,test_missed_rate,maximum_rate_difference,qualified,rank,recommended
0,A,27809,3969,4182,0.458557,0.395062,0.333812,0.124745,True,2,False
1,B,23699,6762,5499,0.460230,0.426205,0.350427,0.109803,True,1,True
2,C,19573,8236,8151,0.441169,0.499879,0.363636,0.136242,True,3,False


## 11. Final boundary decision

In [11]:
TABLES['selected_split_boundaries.csv']

,split_name,start_inclusive,end_exclusive,row_count,row_share,on_time_count,missed_count,missed_target_rate,minimum_created_date,maximum_created_date,selection_reason
0,train,2024-01-01T00:00:00+00:00,2025-04-01T00:00:00+00:00,23699,0.659038,12792,10907,0.460230,2024-01-01 07:58:15+00:00,2025-03-31 23:31:13+00:00,Passes all integrity and minimum-size gates. S...
1,validation,2025-04-01T00:00:00+00:00,2025-09-01T00:00:00+00:00,6762,0.188042,3880,2882,0.426205,2025-04-01 01:22:19+00:00,2025-08-31 23:49:25+00:00,Passes all integrity and minimum-size gates. S...
2,test,2025-09-01T00:00:00+00:00,2026-01-01T00:00:00+00:00,5499,0.152920,3572,1927,0.350427,2025-09-01 04:49:31+00:00,2025-12-31 20:39:09+00:00,Passes all integrity and minimum-size gates. S...


## 12. Split assignment

In [12]:
TABLES['split_row_counts.csv']

,split_name,row_count,row_share
0,train,23699,0.659038
1,validation,6762,0.188042
2,test,5499,0.152920


## 13. Train profile

In [13]:
pd.Series({'rows': len(result.frames.train), 'minimum': result.frames.train.created_date.min(), 'maximum': result.frames.train.created_date.max(), 'missed_rate': float(result.frames.train.missed_resolution_target.mean())})

rows                               23699
minimum        2024-01-01 07:58:15+00:00
maximum        2025-03-31 23:31:13+00:00
missed_rate                      0.46023
dtype: object

## 14. Validation profile

In [14]:
pd.Series({'rows': len(result.frames.validation), 'minimum': result.frames.validation.created_date.min(), 'maximum': result.frames.validation.created_date.max(), 'missed_rate': float(result.frames.validation.missed_resolution_target.mean())})

rows                                6762
minimum        2025-04-01 01:22:19+00:00
maximum        2025-08-31 23:49:25+00:00
missed_rate                     0.426205
dtype: object

## 15. Test profile

In [15]:
pd.Series({'rows': len(result.frames.test), 'minimum': result.frames.test.created_date.min(), 'maximum': result.frames.test.created_date.max(), 'missed_rate': float(result.frames.test.missed_resolution_target.mean())})

rows                                5499
minimum        2025-09-01 04:49:31+00:00
maximum        2025-12-31 20:39:09+00:00
missed_rate                     0.350427
dtype: object

## 16. Chronological integrity

In [16]:
TABLES['split_integrity_checks.csv'].query("area == 'chronology'")

,check_id,area,status,observed_value,expected_value,affected_rows,message
9,chronology.train_range,chronology,PASS,0,0,0,train rows satisfy their configured half-open ...
13,chronology.validation_range,chronology,PASS,0,0,0,validation rows satisfy their configured half-...
17,chronology.test_range,chronology,PASS,0,0,0,test rows satisfy their configured half-open r...
21,chronology.train_before_validation,chronology,PASS,2025-03-31 23:31:13+00:00,2025-04-01 01:22:19+00:00,0,Train observations precede validation observat...
22,chronology.validation_before_test,chronology,PASS,2025-08-31 23:49:25+00:00,2025-09-01 04:49:31+00:00,0,Validation observations precede test observati...


## 17. Identifier-overlap checks

In [17]:
TABLES['identifier_overlap_checks.csv']

,left_split,right_split,overlap_count,status
0,train,validation,0,PASS
1,train,test,0,PASS
2,validation,test,0,PASS


## 18. Target-distribution comparison

In [18]:
TABLES['split_target_distribution.csv']

,split_name,on_time_count,missed_count,total_count,missed_target_rate
0,train,12792,10907,23699,0.460230
1,validation,3880,2882,6762,0.426205
2,test,3572,1927,5499,0.350427


## 19. Temporal-drift summary

In [19]:
TABLES['temporal_drift_summary.csv']

,comparison_type,source_period,destination_period,source_target_rate,destination_target_rate,absolute_difference,relative_difference,source_rows,destination_rows,metric_value,interpretation
0,split_pair,train,validation,0.46023,0.426205,0.034025,0.073931,23699,6762,<NA>,Temporal target prevalence differs; retain as ...
1,split_pair,train,test,0.46023,0.350427,0.109803,0.238583,23699,5499,<NA>,Temporal target prevalence differs; retain as ...
2,split_pair,validation,test,0.426205,0.350427,0.075778,0.177797,6762,5499,<NA>,Temporal target prevalence differs; retain as ...
3,monthly_summary,minimum_monthly_target_rate,2025-11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.307754,Observed monthly instability; not an automatic...
4,monthly_summary,maximum_monthly_target_rate,2025-01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.610039,Observed monthly instability; not an automatic...
5,monthly_summary,monthly_target_rate_range,max minus min,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.302285,Observed monthly instability; not an automatic...
6,monthly_summary,largest_month_to_month_absolute_change,2024-12,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.215554,Observed monthly instability; not an automatic...


## 20. Output reconciliation

In [20]:
TABLES['output_reconciliation.csv']

,check_name,left_value,right_value,status
0,input_equals_train_validation_test,35960,35960,PASS
1,all_rows_assigned,0,0,PASS
2,no_rows_multiply_assigned,0,0,PASS
3,target_counts_reconcile,35960,35960,PASS
4,eligible_source_hash_unchanged,PASS,PASS,PASS
5,eligible_source_mtime_unchanged,PASS,PASS,PASS


## 21. Split metadata

In [21]:
pd.Series(result.metadata.to_dict())

schema_version                                                                       1.0
split_id                                               20260804T170340Z_9d945cb2da0eecfc
completion_status                                                                success
created_at_utc                                          2026-08-04T17:03:40.162054+00:00
source_cleaning_run_id                                 20260804T161733Z_560ff9d5f0dd38f7
source_cleaning_metadata_path          data/processed/resolution_risk/run_id=20260804...
source_eligible_dataset_path           data/processed/resolution_risk/run_id=20260804...
source_eligible_sha256                 8f3b24ae342cb32af726d4d191c00e181037d4ae01672c...
source_eligible_mtime                                                1785860255491958215
source_raw_run_id                                      20260731T122433Z_7e3a488efc738a9c
source_raw_sha256                      f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...
scope_agency         

## 22. Source immutability verification

In [22]:
pd.Series({'hash_before': result.source.eligible_sha256, 'hash_after': sha256_file(result.source.eligible_path), 'mtime_before': result.source.eligible_mtime_ns, 'mtime_after': result.source.eligible_path.stat().st_mtime_ns})

hash_before     8f3b24ae342cb32af726d4d191c00e181037d4ae01672c...
hash_after      8f3b24ae342cb32af726d4d191c00e181037d4ae01672c...
mtime_before                                  1785860255491958215
mtime_after                                   1785860255491958215
dtype: object

## 23. Test-set governance

In [23]:
'The test set is untouched by feature selection, preprocessing design, category policy, threshold selection, tuning, and model selection.'

'The test set is untouched by feature selection, preprocessing design, category policy, threshold selection, tuning, and model selection.'

## 24. Known limitations

In [24]:
pd.Series(result.metadata.warnings, name='limitation_or_warning')

0    Target prevalence varies over time; drift is e...
1    Test data is reserved for one final evaluation...
2    Split outputs are governed analytical partitio...
Name: limitation_or_warning, dtype: str

## 25. Step 8 completion decision

In [25]:
assert TABLES['split_integrity_checks.csv']['status'].eq('PASS').all()
assert TABLES['output_reconciliation.csv']['status'].eq('PASS').all()
assert sum(item.recommended for item in result.candidates) == 1
'Step 8 complete: chronological partitions only; no API, preprocessing, feature matrix, evaluation, or model training.'

'Step 8 complete: chronological partitions only; no API, preprocessing, feature matrix, evaluation, or model training.'